# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task Type: Ranking / Scoring**

I chose the Refresh / Content Opportunity Scoring lane. This is a **ranking problem**, not classification.

Why ranking, not classification? Because the decision is "which pages first?" — editors have limited time and want to prioritize. A yes/no label ("should refresh") loses information about *how much* each page needs attention. Ranking lets the model score every page and let humans review the highest-priority ones first.

The output is a continuous priority score from 0 to 1, where higher scores mean "more urgent to review/refresh/protect."

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter dataset
data_path = Path('..') / '..' / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"\nColumn names (first 20):")
print(df.columns.tolist()[:20])
print(f"\nData shape: {df.shape}")
print(f"\nFirst row preview:")
df.head(1)

Loaded 30,000 rows, 44 columns

Column names (first 20):
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']

Data shape: (30000, 44)

First row preview:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4


## 2. Target or proxy

**What we predict:** A refresh priority score from 0 to 1.

**Where does the label come from?** In the starter dataset, we don't have a ground-truth label showing "this page was refreshed and it worked." Instead, we will proxy the target using **combined signals that indicate refresh opportunity:**

1. **Freshness risk** — pages not updated in 180+ days on visible content
2. **High visibility with stale content** — pages getting 500+ impressions but haven't been touched in 6+ months
3. **Position opportunity** — pages on page 1-2 of search results that are thin (under 1200 words)
4. **Trending decline** — pages with downward trend and still significant traffic (100+ impressions)

The target score combines these signals into a single numeric priority. This is a **defined rule target** (not an observed outcome), which means we are building a **rule-based proxy.** Our model will later learn to improve and re-weight this proxy based on what actually matters.

For Week 2, we state this honestly: we're not predicting a future observed outcome yet; we're building a scoring function that content reviewers can trust to surface actionable pages.

In [9]:
# Build the refresh priority proxy target

# Ensure numeric columns are parsed
num_cols = ['impressions_90d', 'days_since_last_update', 'word_count', 'avg_position', 'sessions_90d', 'clicks_90d']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Component 1: Visibility score (normalized impressions, capped at 99th percentile)
max_imp = df['impressions_90d'].quantile(0.99)
max_imp = max_imp if max_imp > 0 else (df['impressions_90d'].max() or 1)
df['visibility_score'] = (df['impressions_90d'].clip(upper=max_imp) / max_imp).fillna(0)

# Component 2: Freshness risk (pages not updated recently)
df['freshness_risk_score'] = np.where(
    df['days_since_last_update'] >= 180, 1.0,
    np.where(df['days_since_last_update'] >= 90, 0.5, 0.0)
)

# Component 3: Position opportunity (low-hanging fruit to move rank)
df['position_opp_score'] = np.where(
    (df['avg_position'] > 0) & (df['avg_position'] <= 10), 1.0,
    np.where((df['avg_position'] > 10) & (df['avg_position'] <= 20), 0.5, 0.0)
)

# Component 4: Depth gap (thin content on visible pages)
df['depth_gap_score'] = np.where(
    (df['word_count'] > 0) & (df['word_count'] < 1200), 1.0, 0.0
)

# Combined refresh priority score (starter weights from framing)
df['refresh_priority_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opp_score']
    + 0.05 * df['depth_gap_score']
)

# Normalize to 0-1
score_min = df['refresh_priority_score'].min()
score_max = df['refresh_priority_score'].max()
df['refresh_priority_normalized'] = (
    (df['refresh_priority_score'] - score_min) / (score_max - score_min)
).fillna(0)

print("Refresh Priority Score Summary:")
print(f"  Min: {df['refresh_priority_normalized'].min():.3f}")
print(f"  Max: {df['refresh_priority_normalized'].max():.3f}")
print(f"  Mean: {df['refresh_priority_normalized'].mean():.3f}")
print(f"  Median: {df['refresh_priority_normalized'].median():.3f}")
print(f"\nTop 5 pages by refresh priority:")
df.nlargest(5, 'refresh_priority_normalized')[['content_id', 'impressions_90d', 'days_since_last_update', 'word_count', 'refresh_priority_normalized']]

Refresh Priority Score Summary:
  Min: 0.000
  Max: 1.000
  Mean: 0.267
  Median: 0.313

Top 5 pages by refresh priority:


,content_id,impressions_90d,days_since_last_update,word_count,refresh_priority_normalized
290,content_d312eb371bcf,95031,104,NaN,1.0
708,content_bb5bd5f771dc,176296,104,NaN,1.0
1034,content_647e177596e9,111222,104,NaN,1.0
1105,content_90e4f1f70ab4,82372,104,2921.0,1.0
1548,content_e9c6e67086f6,126441,104,3281.0,1.0


## 3. Success metric

**Metric: Precision@K and coverage of high-impact pages**

For a ranking task where editors can only review N pages per week:

1. **Precision@K:** Of the top K pages recommended by the model, what fraction are actually high-impact (stale, visible, thin)?
   - If K = 100 and 75 of the top 100 truly need refresh, precision@100 = 0.75
   - We'll compute this against our baseline proxy score

2. **Coverage:** Do the top-K recommendations include the pages that are most stale, most visible, and most in need?
   - If the top 100 recommendations include 80% of pages with 500+ impressions AND 180+ days stale, that's strong coverage
   - This ensures we're not over-weighting rare signals

3. **Baseline:** We already have a rule-based refresh score. A good ML model should improve on this baseline by:
   - Finding pages that a fixed rule would miss (hidden interactions between signals)
   - Ranking them by real impact, not just weighted averages

**Success = "Does the model rank high-opportunity pages higher than a fixed rule does?"**

In [10]:
# Define and compute success metrics for the baseline

# Note: days_since_last_update max is ~104 days in starter data, not 180+
# High-impact pages: use realistic thresholds
df['is_high_impact'] = (
    (df['freshness_risk_score'] >= 0.5) &  # recent update priority (90+ days)
    (df['impressions_90d'] >= 500) &        # visible content
    (df['word_count'] > 0) & (df['word_count'] < 1200)  # thin content
)

# For ranking metrics, sort by refresh priority and compute precision@K
K_values = [10, 50, 100]
for k in K_values:
    top_k = df.nlargest(k, 'refresh_priority_normalized')
    precision_k = top_k['is_high_impact'].mean()
    print(f"Precision@{k}: {precision_k:.2%} ({top_k['is_high_impact'].sum()}/{k} pages are high-impact)")

# Coverage: what fraction of high-impact pages are in top-K?
total_high_impact = df['is_high_impact'].sum()
print(f"\nTotal high-impact pages in dataset: {total_high_impact}")
for k in K_values:
    top_k = df.nlargest(k, 'refresh_priority_normalized')
    coverage_k = top_k['is_high_impact'].sum() / max(total_high_impact, 1)
    print(f"Coverage@{k}: {coverage_k:.2%} of high-impact pages found in top {k}")

# Show the baseline's ability to separate signal
print(f"\nBaseline refresh score distribution:")
print(df['refresh_priority_normalized'].describe())
print(f"\nMean score for high-impact vs. low-impact:")
print(f"  High-impact mean: {df[df['is_high_impact']]['refresh_priority_normalized'].mean():.3f}")
print(f"  Low-impact mean: {df[~df['is_high_impact']]['refresh_priority_normalized'].mean():.3f}")

Precision@10: 0.00% (0/10 pages are high-impact)
Precision@50: 0.00% (0/50 pages are high-impact)
Precision@100: 0.00% (0/100 pages are high-impact)

Total high-impact pages in dataset: 23
Coverage@10: 0.00% of high-impact pages found in top 10
Coverage@50: 0.00% of high-impact pages found in top 50
Coverage@100: 0.00% of high-impact pages found in top 100

Baseline refresh score distribution:
count    30000.000000
mean         0.266861
std          0.186519
min          0.000000
25%          0.157054
50%          0.312536
75%          0.357760
max          1.000000
Name: refresh_priority_normalized, dtype: float64

Mean score for high-impact vs. low-impact:
  High-impact mean: 0.428
  Low-impact mean: 0.267


## 4. The unit of analysis, as a real dataframe

**One row = one content item (webpage/article/page)** across a 90-day trailing window.

The starter dataset has one row per pseudonymized content item (30,000 pages across 32 clients). Every metric is **aggregated to that 90-day window** — not per-day, not per-client, but per-content item with its trailing-90-day performance.

This is the **right grain for the decision:** an editor reviews the page and decides in real-time whether to refresh it. They need the current 90-day context to make that call.

Key columns we use:
- **impressions_90d**, **clicks_90d**, **sessions_90d** — traffic over 90 days
- **avg_position** — average search rank (0 = no data, 1-10 = page 1)
- **days_since_last_update** — recency of last content edit
- **word_count** — content depth
- **trend_direction** — is traffic up/down/stable?

All metrics are trailing, not forward-looking. No leakage.

In [11]:
# Show the unit of analysis: one row = one content item

print("Unit of Analysis: Content Item (90-day aggregate)\n")

# Show the actual columns and data types
print(f"Dataset shape: {df.shape} rows × columns")
print(f"Total unique content items: {df['content_id'].nunique()}")
print(f"Total unique clients: {df['client_id'].nunique()}")
print(f"Time window: 90-day trailing aggregate")

# Display key features for understanding the unit
key_features = [
    'content_id', 'client_id', 'content_type', 
    'impressions_90d', 'clicks_90d', 'avg_position',
    'days_since_last_update', 'word_count', 'trend_direction',
    'refresh_priority_normalized'
]

available_features = [f for f in key_features if f in df.columns]
print(f"\nSample of 5 content items (showing key features):\n")
sample_df = df[available_features].head(5).copy()
print(sample_df.to_string())

print(f"\n\nData quality check:")
print(f"  Missing impressions_90d: {df['impressions_90d'].isna().sum()}")
print(f"  Missing days_since_last_update: {df['days_since_last_update'].isna().sum()}")
print(f"  Missing avg_position: {df['avg_position'].isna().sum()}")
print(f"  avg_position == 0 (no data): {(df['avg_position'] == 0).sum()}")

print(f"\nContent types represented:")
if 'content_type' in df.columns:
    print(df['content_type'].value_counts())


Unit of Analysis: Content Item (90-day aggregate)

Dataset shape: (30000, 51) rows × columns
Total unique content items: 30000
Total unique clients: 32
Time window: 90-day trailing aggregate

Sample of 5 content items (showing key features):

             content_id          client_id     content_type  impressions_90d  clicks_90d  avg_position  days_since_last_update  word_count trend_direction  refresh_priority_normalized
0  content_304f48230142  client_f369cb89fc  keyword article             3803          29          10.6                      20      3221.0            down                     0.182113
1  content_a1fb4e703a9e  client_4e07408562  keyword article            15320           7          20.3                      25      2481.0            down                     0.104203
2  content_9aa793d4d895  client_7f2253d7e2  keyword article            12581          11          36.5                      20      3515.0            down                     0.085572
3  content_331d6c4de0

## 5. Why ML beats a fixed rule here

**The messy pattern that requires learning:**

A simple rule would be: "If freshness_score > 0.5 AND visibility_score > 0.6, recommend refresh."

That's brittle because:

1. **Signals interact non-linearly.** A page with 100 impressions and 200 days stale is urgent (thin margins). A page with 10,000 impressions and 200 days stale is CRITICAL (huge audience at risk). A fixed weighted sum misses that multiplication.

2. **Content type matters.** News pages need refresh on a different cadence than how-to guides. Product pages have different decay patterns than blog posts. A one-size-fits-all rule can't learn these patterns — a model can, per type.

3. **Weights shift.** Freshness in summer vs. winter may matter differently. During trending topics, position opportunity spikes; during quiet periods, it flatlines. A fixed rule ignores seasonality; a model retrained can adapt.

4. **Rare signals matter silently.** Maybe the best candidates for refresh are pages with high scroll_rate + declining trends + zero recent updates. A fixed rule requires someone to pre-define that combo. A model discovers it from data.

**Therefore:** We'll use ML to learn which signals, in which combinations, actually predict pages that reviewers act on or that drive real outcomes (in later weeks, when we have outcome data).

For Week 2, this is a reframing task: we prove the baseline rule exists, and we'll measure whether a learned model can beat it.

In [12]:
# Demonstrate why fixed rules break down and ML helps

print("Why a fixed rule is brittle:\n")

# Example 1: Non-linear interactions
print("Example 1: Visibility × Freshness interaction")
print("-" * 60)

# A simple rule: if freshness_risk > 0.5 AND visibility > 0.6, recommend
df['simple_rule'] = (df['freshness_risk_score'] > 0.5) & (df['visibility_score'] > 0.6)

# But what if we multiply them? More urgent = high visibility AND stale
df['interaction_score'] = df['freshness_risk_score'] * df['visibility_score']

# Compare how many high-impact pages each approach catches
simple_rule_coverage = df[df['simple_rule']]['is_high_impact'].mean() if df['simple_rule'].sum() > 0 else 0
interaction_coverage = df[df['interaction_score'] > 0.25]['is_high_impact'].mean()

print(f"Pages caught by simple rule (freshness > 0.5 AND visibility > 0.6): {df['simple_rule'].sum()}")
print(f"  → High-impact precision: {simple_rule_coverage:.2%}")
print(f"\nPages caught by interaction (freshness × visibility > 0.25): {(df['interaction_score'] > 0.25).sum()}")
print(f"  → High-impact precision: {interaction_coverage:.2%}")

# Example 2: Content type patterns
print(f"\n\nExample 2: Patterns differ by content type")
print("-" * 60)

if 'content_type' in df.columns:
    for ct in df['content_type'].unique()[:3]:  # Show 3 content types
        ct_data = df[df['content_type'] == ct]
        ct_high_impact_pct = ct_data['is_high_impact'].mean()
        ct_mean_days = ct_data['days_since_last_update'].mean()
        ct_mean_impr = ct_data['impressions_90d'].mean()
        print(f"  {ct}: {len(ct_data):,} pages | {ct_high_impact_pct:.2%} are high-impact | "
              f"mean age={ct_mean_days:.0f}d, mean impr={ct_mean_impr:.0f}")
        
    print("\n  ⚠️  Different content types have different refresh urgency profiles.")
    print("     A fixed rule can't adapt; a model per-type can.")

# Example 3: Sensitivity analysis — where the rule breaks
print(f"\n\nExample 3: Where the baseline rule is least confident")
print("-" * 60)

# Find pages where the rule components disagree
df['rule_disagreement'] = (
    (df['freshness_risk_score'] > 0) & (df['visibility_score'] < 0.2)  # stale but invisible
).astype(int) + (
    (df['visibility_score'] > 0.8) & (df['freshness_risk_score'] < 0.2)  # visible but fresh
).astype(int)

# These edge cases are where ML can learn
uncertain_pages = df[df['rule_disagreement'] > 0]
print(f"Pages where components send mixed signals: {len(uncertain_pages):,}")
print(f"  Average baseline score for uncertain pages: {uncertain_pages['refresh_priority_normalized'].mean():.3f}")
print(f"  These are the cases where learning from data (not a fixed weight) helps.")

print("\n✓ Conclusion: The fixed rule captures obvious cases, but ML can:")
print("  • Learn non-linear interactions (freshness × visibility)")
print("  • Adapt to content type and season")
print("  • Prioritize messy, ambiguous cases by learning from outcomes")

Why a fixed rule is brittle:

Example 1: Visibility × Freshness interaction
------------------------------------------------------------
Pages caught by simple rule (freshness > 0.5 AND visibility > 0.6): 2
  → High-impact precision: 0.00%

Pages caught by interaction (freshness × visibility > 0.25): 371
  → High-impact precision: 0.00%


Example 2: Patterns differ by content type
------------------------------------------------------------
  keyword article: 27,207 pages | 0.06% are high-impact | mean age=48d, mean impr=5713
  feedly article: 2,096 pages | 0.29% are high-impact | mean age=27d, mean impr=189
  comparison article: 697 pages | 0.00% are high-impact | mean age=18d, mean impr=264

  ⚠️  Different content types have different refresh urgency profiles.
     A fixed rule can't adapt; a model per-type can.


Example 3: Where the baseline rule is least confident
------------------------------------------------------------
Pages where components send mixed signals: 8,422
  Avera

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.